In [0]:
import urllib.request
import os
import shutil

#### EXTRACT ALL HISTORICAL YELLOW TAXI TRIP FOR YEAR - 2025 FROM API TO ADLS

In [0]:
months_to_process = ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', 
                     '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12']

for months in months_to_process:
    url = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{months}.parquet'
    response = urllib.request.urlopen(url)

    dir_path = f'/Volumes/nyctaxi/landing/operational/2025/YellowTaxi/{months}'
    os.makedirs(dir_path, exist_ok = True)

    local_file_path = dir_path + f'/yellow_tripdata_{months}.parquet'
    with open(local_file_path, 'wb') as f:
        shutil.copyfileobj(response, f)

#### LOAD HISTORICAL YELLOW TAXI 
- `NYCTAXI.BRONZE.YELLOW_TAXI`

In [0]:
from pyspark.sql.functions import input_file_name, col, current_timestamp

yellow_taxi_df = (spark.read.format('parquet')
                      .load('/Volumes/nyctaxi/landing/operational/2025/YellowTaxi/*')
                      .withColumn('file_name', col('_metadata.file_path'))
                      .withColumn('load_time_stamp', current_timestamp())
                )
display(yellow_taxi_df.limit(1))

In [0]:
yellow_taxi_df.write.mode('append').saveAsTable('nyctaxi.bronze.yellow_taxi')

In [0]:
dbutils.notebook.exit('HISTORICAL NYC YELLOW TRIP FILES HAS BEEN LOADED INTO NYCTAXI.BRONZE.YELLOW_TAXI')